### ORM(object Relational Mapping)
- database table => python class (1:1 mapping)
- 컬럼 == 속성
- SQLAlchemy 라이브러리가 ORM 지원

In [1]:
!pip3 install sqlalchemy

In [7]:
# User 테이블(id, name, email)
# create table users()
from sqlalchemy import Column, Integer, String, create_engine
from sqlalchemy.orm import declarative_base,sessionmaker

# 데이터 베이스 연결
# 애플리케이션 당 하나만 만들어 사용
# echo=True: 실행 SQL을 콘솔에 출력
engine = create_engine('sqlite:///db/users.db',echo=True)

# 모든 모델 클래스의 부모 클래스가될 Base 객체 생성
Base=declarative_base()

# 데이터 베이스랑 관련있는 클래스 = 모델 클래스
class User(Base):
    # 테이블 이름 지정
    __tablename__ = "users"

    id=Column(Integer,primary_key=True)
    name=Column(String)
    email=Column(String, unique=True)

    def __str__(self):
        return f"<User(name='{self.name}',email='{self.email}')>"

# 테이블 생성(없는경우에만 새로 생성)
Base.metadata.create_all(engine)


2026-06-17 11:34:36,594 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 11:34:36,595 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("users")
2026-06-17 11:34:36,596 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-17 11:34:36,597 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("users")
2026-06-17 11:34:36,598 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-17 11:34:36,599 INFO sqlalchemy.engine.Engine 
CREATE TABLE users (
	id INTEGER NOT NULL, 
	name VARCHAR, 
	email VARCHAR, 
	PRIMARY KEY (id), 
	UNIQUE (email)
)


2026-06-17 11:34:36,600 INFO sqlalchemy.engine.Engine [no key 0.00117s] ()
2026-06-17 11:34:36,608 INFO sqlalchemy.engine.Engine COMMIT


In [8]:
Base.metadata.drop_all(engine)

2026-06-17 11:34:40,130 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 11:34:40,131 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("users")
2026-06-17 11:34:40,131 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-17 11:34:40,132 INFO sqlalchemy.engine.Engine 
DROP TABLE users
2026-06-17 11:34:40,132 INFO sqlalchemy.engine.Engine [no key 0.00076s] ()
2026-06-17 11:34:40,135 INFO sqlalchemy.engine.Engine COMMIT


In [2]:
from sqlalchemy import text

engine = create_engine('sqlite:///db/demo.db',echo=True)
with engine.connect() as conn:
    conn.execute(text('''
    CREATE TABLE IF NOT EXISTS users(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        age INTEGER              
        )
'''))
    conn.commit()
    conn.execute(text("INSERT INTO users(name, age) VALUES(:name, :age)"),
                 [{"name":"Alice", "age":30},{"name":"Bob", "age":25}])
    conn.commit()

    result=conn.execute(text("SELECT * FROM users"))
    for row in result:
        print(row.id, row.name, row.age)

2026-06-17 11:33:56,842 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 11:33:56,842 INFO sqlalchemy.engine.Engine 
    CREATE TABLE IF NOT EXISTS users(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        age INTEGER              
        )

2026-06-17 11:33:56,842 INFO sqlalchemy.engine.Engine [generated in 0.00155s] ()
2026-06-17 11:33:56,842 INFO sqlalchemy.engine.Engine COMMIT
2026-06-17 11:33:56,845 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 11:33:56,845 INFO sqlalchemy.engine.Engine INSERT INTO users(name, age) VALUES(?, ?)
2026-06-17 11:33:56,845 INFO sqlalchemy.engine.Engine [generated in 0.00129s] [('Alice', 30), ('Bob', 25)]
2026-06-17 11:33:56,845 INFO sqlalchemy.engine.Engine COMMIT
2026-06-17 11:33:56,858 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 11:33:56,859 INFO sqlalchemy.engine.Engine SELECT * FROM users
2026-06-17 11:33:56,860 INFO sqlalchemy.engine.Engine [generated in 0.00161s] ()
1 Alice 30
2 B

In [9]:
from sqlalchemy.orm import DeclarativeBase,Mapped,mapped_column
from sqlalchemy import DateTime, func
from datetime import datetime

Base=declarative_base()

# 데이터 베이스랑 관련있는 클래스 = 모델 클래스
class Base(DeclarativeBase):
    pass

class User(Base):
    __tablename__ = "users"

    id:Mapped[int] = mapped_column(primary_key=True,autoincrement=True)
    name:Mapped[str]
    email:Mapped[str] = mapped_column(unique=True)
    age:Mapped[int]
    created_at:Mapped[datetime]=mapped_column(DateTime,server_default=func.now())

    def __str__(self):
        return f"<User(name='{self.name}',email='{self.email}')>"

# engine 관련 모든 테이블 삭제
Base.metadata.create_all(engine)


2026-06-17 11:34:46,514 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 11:34:46,515 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("users")
2026-06-17 11:34:46,516 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-17 11:34:46,518 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("users")
2026-06-17 11:34:46,518 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-17 11:34:46,518 INFO sqlalchemy.engine.Engine 
CREATE TABLE users (
	id INTEGER NOT NULL, 
	name VARCHAR NOT NULL, 
	email VARCHAR NOT NULL, 
	age INTEGER NOT NULL, 
	created_at DATETIME DEFAULT CURRENT_TIMESTAMP NOT NULL, 
	PRIMARY KEY (id), 
	UNIQUE (email)
)


2026-06-17 11:34:46,518 INFO sqlalchemy.engine.Engine [no key 0.00155s] ()
2026-06-17 11:34:46,525 INFO sqlalchemy.engine.Engine COMMIT


#### 세션
- 데이터 베이스 연동
- 변경 사항 추적, 트랜잭션 관리
- sessionMaker 팩토리 사용

In [10]:
# tptus vorxhfl
Session = sessionmaker(bind=engine, autoflush=False, autocommit=False)
# tptusrorcp
session = Session()

In [11]:
session.close()

In [12]:
#insert
# session.add()/session.add_all()/session.commit()
session = Session()
# 사용자 생성
new_user = User( name="Alice", email="alice@example.com",age=35)
session.add(new_user)
session.commit()
session.close()

2026-06-17 11:34:58,886 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 11:34:58,888 INFO sqlalchemy.engine.Engine INSERT INTO users (name, email, age) VALUES (?, ?, ?) RETURNING id, created_at
2026-06-17 11:34:58,889 INFO sqlalchemy.engine.Engine [generated in 0.00092s] ('Alice', 'alice@example.com', 35)
2026-06-17 11:34:58,892 INFO sqlalchemy.engine.Engine COMMIT


In [13]:
Session = sessionmaker(bind=engine, autoflush=False, autocommit=False)

# --- [신규] 여러 명의 데이터를 리스트로 준비 ---
user_list = [
    User(name="Bob", email="bob@example.com", age=25),
    User(name="Charlie", email="charlie@example.com", age=28),
    User(name="David", email="david@example.com", age=32)
]

# --- with 문 실행 ---
with Session() as session:
    try:
        # ⭐ add_all()을 사용하여 리스트에 담긴 모든 객체를 한 번에 세션에 올립니다.
        session.add_all(user_list)
        
        # 단 한 번의 커밋으로 모든 유저 정보를 데이터베이스에 반영합니다.
        session.commit()
        print(f"🎉 성공적으로 {len(user_list)}명의 유저를 일괄 등록했습니다!")
        
    except Exception as e:
        # 단 하나라도 에러(예: 이메일 중복 등)가 나면 전체 작업을 취소하고 롤백합니다.
        session.rollback()
        print(f"❌ 다중 등록 실패, 전체 롤백되었습니다: {e}")

2026-06-17 11:35:02,469 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 11:35:02,471 INFO sqlalchemy.engine.Engine INSERT INTO users (name, email, age) VALUES (?, ?, ?) RETURNING id, created_at
2026-06-17 11:35:02,472 INFO sqlalchemy.engine.Engine [generated in 0.00008s (insertmanyvalues) 1/3 (ordered; batch not supported)] ('Bob', 'bob@example.com', 25)
2026-06-17 11:35:02,473 INFO sqlalchemy.engine.Engine INSERT INTO users (name, email, age) VALUES (?, ?, ?) RETURNING id, created_at
2026-06-17 11:35:02,475 INFO sqlalchemy.engine.Engine [insertmanyvalues 2/3 (ordered; batch not supported)] ('Charlie', 'charlie@example.com', 28)
2026-06-17 11:35:02,475 INFO sqlalchemy.engine.Engine INSERT INTO users (name, email, age) VALUES (?, ?, ?) RETURNING id, created_at
2026-06-17 11:35:02,475 INFO sqlalchemy.engine.Engine [insertmanyvalues 3/3 (ordered; batch not supported)] ('David', 'david@example.com', 32)
2026-06-17 11:35:02,475 INFO sqlalchemy.engine.Engine COMMIT
🎉 성공적으로 3명의 유저를 

In [14]:
from sqlalchemy import select

with Session() as session:
    # id조회
    user = session.get(User, 1)
    print("""----단일 사용자----""")
    print(user)

    print("""----모든 사용자----""")
    all_users = session.query(User).all()
    for user in all_users:
        print(user)
        print(user.id)

    stmt = select(User)
    all_users = session.scalars(stmt).all()
    for user in all_users:
        print(user)

    print("""----where----""")
    alice = session.query(User).filter_by(name='Alice').first()
    print(f"찾는 사람 {alice}")

    stmt = select(User).where(User.age >= 30)
    find_users = session.scalars(stmt).all()
    for user in find_users:
        print(user)

    stmt = select(User).where(User.email.like('%ali%'))
    find_users = session.scalars(stmt).all()
    for user in find_users:
        print(user)
    
    stmt = select(User).order_by(User.id.desc()).limit(2)
    find_users = session.scalars(stmt).all()
    for user in find_users:
        print(user)
    
    count = session.scalars(select(func.count()).select_from(User))
    
    print(count)

2026-06-17 11:35:07,044 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 11:35:07,049 INFO sqlalchemy.engine.Engine SELECT users.id AS users_id, users.name AS users_name, users.email AS users_email, users.age AS users_age, users.created_at AS users_created_at 
FROM users 
WHERE users.id = ?
2026-06-17 11:35:07,049 INFO sqlalchemy.engine.Engine [generated in 0.00109s] (1,)
----단일 사용자----
<User(name='Alice',email='alice@example.com')>
----모든 사용자----
2026-06-17 11:35:07,053 INFO sqlalchemy.engine.Engine SELECT users.id AS users_id, users.name AS users_name, users.email AS users_email, users.age AS users_age, users.created_at AS users_created_at 
FROM users
2026-06-17 11:35:07,053 INFO sqlalchemy.engine.Engine [generated in 0.00071s] ()
<User(name='Alice',email='alice@example.com')>
1
<User(name='Bob',email='bob@example.com')>
2
<User(name='Charlie',email='charlie@example.com')>
3
<User(name='David',email='david@example.com')>
4
2026-06-17 11:35:07,056 INFO sqlalchemy.engine.Engin

In [15]:
with Session() as session:
    alice = session.query(User).filter_by(name="Alice").first()
    alice.email = 'alice.new@exemple.com'
    session.commit()

    print(f"수정된 사용자 {session.query(User).filter_by(name="Alice").first()}")

2026-06-17 11:35:09,697 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 11:35:09,698 INFO sqlalchemy.engine.Engine SELECT users.id AS users_id, users.name AS users_name, users.email AS users_email, users.age AS users_age, users.created_at AS users_created_at 
FROM users 
WHERE users.name = ?
 LIMIT ? OFFSET ?
2026-06-17 11:35:09,698 INFO sqlalchemy.engine.Engine [cached since 2.639s ago] ('Alice', 1, 0)
2026-06-17 11:35:09,702 INFO sqlalchemy.engine.Engine UPDATE users SET email=? WHERE users.id = ?
2026-06-17 11:35:09,702 INFO sqlalchemy.engine.Engine [generated in 0.00142s] ('alice.new@exemple.com', 1)
2026-06-17 11:35:09,702 INFO sqlalchemy.engine.Engine COMMIT
2026-06-17 11:35:09,705 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 11:35:09,705 INFO sqlalchemy.engine.Engine SELECT users.id AS users_id, users.name AS users_name, users.email AS users_email, users.age AS users_age, users.created_at AS users_created_at 
FROM users 
WHERE users.name = ?
 LIMIT ? OFFSE

In [16]:
from sqlalchemy import update

with Session() as session:
    user = session.get(User, 1)
    if user:
        user.age = 40
        session.commit()

    stmt = update(User).where(User.age<=30).values(age=20)
    session.execute(stmt)
    session.commit()

2026-06-17 11:35:11,517 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 11:35:11,519 INFO sqlalchemy.engine.Engine SELECT users.id AS users_id, users.name AS users_name, users.email AS users_email, users.age AS users_age, users.created_at AS users_created_at 
FROM users 
WHERE users.id = ?
2026-06-17 11:35:11,519 INFO sqlalchemy.engine.Engine [cached since 4.471s ago] (1,)
2026-06-17 11:35:11,520 INFO sqlalchemy.engine.Engine UPDATE users SET age=? WHERE users.id = ?
2026-06-17 11:35:11,521 INFO sqlalchemy.engine.Engine [generated in 0.00048s] (40, 1)
2026-06-17 11:35:11,523 INFO sqlalchemy.engine.Engine COMMIT
2026-06-17 11:35:11,525 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 11:35:11,533 INFO sqlalchemy.engine.Engine UPDATE users SET age=? WHERE users.age <= ?
2026-06-17 11:35:11,535 INFO sqlalchemy.engine.Engine [generated in 0.00066s] (20, 30)
2026-06-17 11:35:11,536 INFO sqlalchemy.engine.Engine COMMIT


In [17]:

with Session() as session:
    bob = session.query(User).filter_by(name="Bob").first()
    session.delete(bob)
    session.commit()

2026-06-17 11:35:14,092 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 11:35:14,093 INFO sqlalchemy.engine.Engine SELECT users.id AS users_id, users.name AS users_name, users.email AS users_email, users.age AS users_age, users.created_at AS users_created_at 
FROM users 
WHERE users.name = ?
 LIMIT ? OFFSET ?
2026-06-17 11:35:14,093 INFO sqlalchemy.engine.Engine [cached since 7.034s ago] ('Bob', 1, 0)
2026-06-17 11:35:14,096 INFO sqlalchemy.engine.Engine DELETE FROM users WHERE users.id = ?
2026-06-17 11:35:14,096 INFO sqlalchemy.engine.Engine [generated in 0.00100s] (2,)
2026-06-17 11:35:14,100 INFO sqlalchemy.engine.Engine COMMIT


In [18]:
from sqlalchemy import delete

# delete form 테이블명 where id=1

with Session() as session:
    stmt = delete(User).where(User.id==2)
    session.execute(stmt)
    session.commit()

2026-06-17 11:35:15,796 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 11:35:15,798 INFO sqlalchemy.engine.Engine DELETE FROM users WHERE users.id = ?
2026-06-17 11:35:15,799 INFO sqlalchemy.engine.Engine [generated in 0.00077s] (2,)
2026-06-17 11:35:15,800 INFO sqlalchemy.engine.Engine COMMIT


In [19]:
Base.metadata.clear()

In [20]:
# 관계(외래키)
# 1:N, N:1, M:N
#게시글 하나:댓글 N개
from sqlalchemy import ForeignKey
from sqlalchemy.orm import relationship

class Post(Base):
    __tablename__="posts"

    id:Mapped[int] = mapped_column(primary_key=True,autoincrement=True)
    title:Mapped[str] = mapped_column(String(200))
    content:Mapped[str]

    comments:Mapped[list['Comment']] = relationship('Comment',back_populates='post',cascade='all, delete-orphan')
class Comment(Base):
    __tablename__ = "comments"

    id:Mapped[int] = mapped_column(primary_key=True,autoincrement=True)
    content:Mapped[str]
    # 어느 게시글의 댓글인가
    post_id:Mapped[int]=mapped_column(ForeignKey('posts.id'))
    post: Mapped["Post"] = relationship("Post", back_populates="comments")

In [21]:
Base.metadata.create_all(engine)

2026-06-17 11:51:15,682 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 11:51:15,684 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("posts")
2026-06-17 11:51:15,685 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-17 11:51:15,686 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("posts")
2026-06-17 11:51:15,687 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-17 11:51:15,688 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("comments")
2026-06-17 11:51:15,690 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-17 11:51:15,690 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("comments")
2026-06-17 11:51:15,691 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-17 11:51:15,692 INFO sqlalchemy.engine.Engine 
CREATE TABLE posts (
	id INTEGER NOT NULL, 
	title VARCHAR(200) NOT NULL, 
	content VARCHAR NOT NULL, 
	PRIMARY KEY (id)
)


2026-06-17 11:51:15,693 INFO sqlalchemy.engine.Engine [no key 0.00094s] ()
2026-06-17 11:51:15,702 INFO sqlalchemy.engine.Engine 
C

In [24]:
with Session() as session:
    post = Post(title="LLM 입문",content="LLM이란 무엇인가?")
    post.comments=[
        Comment(content="정말 좋아요"),
        Comment(content="좋은 설명 감사합니다.")
    ]
    session.add(post)
    session.commit()

    p=session.get(Post, 1)
    for c in p.comments:
        print(c.content)

2026-06-17 11:54:45,259 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 11:54:45,261 INFO sqlalchemy.engine.Engine INSERT INTO posts (title, content) VALUES (?, ?)
2026-06-17 11:54:45,262 INFO sqlalchemy.engine.Engine [generated in 0.00104s] ('LLM 입문', 'LLM이란 무엇인가?')
2026-06-17 11:54:45,266 INFO sqlalchemy.engine.Engine INSERT INTO comments (content, post_id) VALUES (?, ?) RETURNING id
2026-06-17 11:54:45,267 INFO sqlalchemy.engine.Engine [generated in 0.00015s (insertmanyvalues) 1/2 (ordered; batch not supported)] ('정말 좋아요', 1)
2026-06-17 11:54:45,268 INFO sqlalchemy.engine.Engine INSERT INTO comments (content, post_id) VALUES (?, ?) RETURNING id
2026-06-17 11:54:45,269 INFO sqlalchemy.engine.Engine [insertmanyvalues 2/2 (ordered; batch not supported)] ('좋은 설명 감사합니다.', 1)
2026-06-17 11:54:45,270 INFO sqlalchemy.engine.Engine COMMIT
2026-06-17 11:54:45,279 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 11:54:45,279 INFO sqlalchemy.engine.Engine SELECT posts.id AS p